# TODO 5 — Benchmark SA : 1 long run vs plusieurs runs courts

## Objectif

Est-il plus efficace de lancer SA **une seule fois avec beaucoup d'itérations**,
ou **plusieurs fois avec peu d'itérations** en gardant la meilleure solution ?

Pour un budget total d'itérations fixé à N :
- **Stratégie A** : 1 run de N itérations (β lent)
- **Stratégie B** : k runs de N/k itérations chacun, on retient le meilleur (β rapide)

On étudie cela sur un pool de 5 instances QUBO de même taille, difficiles à résoudre.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import time

from problem.qubo import QuboProblem
from solver.classical_solver.simulated_annealing import SimulatedAnnealing
from utils.data_loaders import save_qubo_pool, load_qubo_pool
from utils.display_methods import plot_boxplot_comparison

np.random.seed(0)
INSTANCES_DIR = os.path.join('..', 'data', 'benchmark_sa_instances')
print('Imports OK')

## 1. Création et sauvegarde des instances QUBO

On génère 5 instances QUBO aléatoires de taille n=20,
suffisamment difficiles pour que SA ne converge pas systématiquement.
On les sauvegarde pour pouvoir reproduire l'expérience.

In [ ]:
N = 20
N_INSTANCES = 5
rng = np.random.RandomState(99)

instances = []
for i in range(N_INSTANCES):
    Q = np.triu(rng.randint(-10, 10, size=(N, N)).astype(float))
    instances.append(QuboProblem(Q))

save_qubo_pool(instances, INSTANCES_DIR)
print(f'{N_INSTANCES} instances QUBO (n={N}) sauvegardées dans {INSTANCES_DIR}/')

In [ ]:
# Rechargement depuis le disque (vérifie la persistance)
instances = load_qubo_pool(INSTANCES_DIR)
print(f'{len(instances)} instances chargées : {instances}')

## 2. Définition du budget d'itérations

On fixe un budget total de N_TOTAL itérations par instance.
On compare les stratégies :
- **k=1** : 1 run de N_TOTAL itérations
- **k=5** : 5 runs de N_TOTAL/5 itérations chacun
- **k=10** : 10 runs de N_TOTAL/10 itérations chacun
- **k=20** : 20 runs de N_TOTAL/20 itérations chacun

Pour contrôler le nombre d'itérations, on règle β de façon à obtenir
exactement `target_steps` paliers : β = (T_min/T₀)^(1/target_steps)

In [ ]:
T0 = 100.0
T_MIN = 0.01
STEPS_PER_TEMP = 5

# Budget total = nombre de paliers × steps_per_temperature
N_TOTAL_PALIERS = 500   # paliers au total

def beta_for_steps(n_steps, t0=T0, t_min=T_MIN):
    """Calcule β tel que SA effectue exactement n_steps paliers de T₀ à T_min."""
    return (t_min / t0) ** (1.0 / n_steps)

k_values = [1, 2, 5, 10, 20, 50]

for k in k_values:
    steps_per_run = N_TOTAL_PALIERS // k
    beta = beta_for_steps(steps_per_run)
    total_iters = steps_per_run * STEPS_PER_TEMP * k
    print(f'k={k:3d}  steps/run={steps_per_run:5d}  β={beta:.6f}  total_iters={total_iters:7d}')

## 3. Protocole de benchmark

Pour chaque instance, pour chaque stratégie (valeur de k) :
- Répéter 20 fois l'expérience complète (k runs, garder le meilleur)
- Mesurer : valeur QUBO obtenue et temps de calcul total

In [ ]:
N_REPETITIONS = 20  # répétitions de l'expérience complète pour chaque (instance, k)

results = {}  # results[(instance_idx, k)] = {'values': [...], 'times': [...]}

for inst_idx, qubo in enumerate(instances):
    print(f'\n--- Instance {inst_idx} ---')
    for k in k_values:
        steps_per_run = max(1, N_TOTAL_PALIERS // k)
        beta = beta_for_steps(steps_per_run)

        exp_values = []
        exp_times = []

        for rep in range(N_REPETITIONS):
            t_start = time.time()
            best_val = float('inf')
            for _ in range(k):
                sa = SimulatedAnnealing(qubo, maximize=False,
                                        initial_temperature=T0,
                                        cooling_rate=beta,
                                        min_temperature=T_MIN,
                                        steps_per_temperature=STEPS_PER_TEMP)
                val, _ = sa.solve()
                best_val = min(best_val, val)
            exp_values.append(best_val)
            exp_times.append(time.time() - t_start)

        results[(inst_idx, k)] = {'values': exp_values, 'times': exp_times}
        print(f'  k={k:3d}  moy={np.mean(exp_values):8.2f}  std={np.std(exp_values):5.2f}  '
              f'temps_moy={np.mean(exp_times)*1000:.1f}ms')

print('\nBenchmark terminé.')

## 4. Visualisation des résultats

### 4a. Qualité par stratégie (agrégée sur toutes les instances)

In [ ]:
# Agréger les valeurs sur toutes les instances pour chaque k
quality_by_k = {}
time_by_k    = {}

for k in k_values:
    all_vals  = []
    all_times = []
    for inst_idx in range(N_INSTANCES):
        all_vals.extend(results[(inst_idx, k)]['values'])
        all_times.extend(results[(inst_idx, k)]['times'])
    quality_by_k[f'k={k}'] = all_vals
    time_by_k[k] = all_times

fig = plot_boxplot_comparison(quality_by_k,
    title='Qualité selon le nombre de runs k (budget total fixé)',
    xlabel='Stratégie (k runs)',
    ylabel='Meilleure valeur QUBO trouvée')
plt.show()

### 4b. Temps de calcul par stratégie

In [ ]:
mean_times = [np.mean(time_by_k[k]) * 1000 for k in k_values]
std_times  = [np.std(time_by_k[k])  * 1000 for k in k_values]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(k) for k in k_values], mean_times, yerr=std_times,
       color='steelblue', edgecolor='white', capsize=4)
ax.set_xlabel('k (nombre de runs)')
ax.set_ylabel('Temps total moyen (ms)')
ax.set_title('Temps de calcul total selon la stratégie')
plt.tight_layout()
plt.show()

### 4c. Vue par instance — est-ce que les tendances sont cohérentes ?

In [ ]:
fig, axes = plt.subplots(1, N_INSTANCES, figsize=(4 * N_INSTANCES, 4), sharey=True)

for inst_idx, ax in enumerate(axes):
    means = [np.mean(results[(inst_idx, k)]['values']) for k in k_values]
    stds  = [np.std(results[(inst_idx, k)]['values'])  for k in k_values]
    ax.errorbar(k_values, means, yerr=stds, fmt='o-', capsize=4, color='steelblue')
    ax.set_title(f'Instance {inst_idx}')
    ax.set_xlabel('k')
    if inst_idx == 0:
        ax.set_ylabel('Valeur QUBO')

fig.suptitle('Qualité par instance selon k', y=1.02)
plt.tight_layout()
plt.show()

## 5. Analyse statistique

Pour conclure si une stratégie est significativement meilleure,
on compare les moyennes et écarts-types des distributions obtenues.

In [ ]:
print('Résumé statistique agrégé (toutes instances confondues)\n')
print(f'{"k":>5}  {"Moyenne":>10}  {"Std":>8}  {"Min":>8}  {"Max":>8}  {"Temps (ms)":>12}')
print('-' * 60)

for k in k_values:
    all_vals  = [v for inst in range(N_INSTANCES) for v in results[(inst, k)]['values']]
    all_times = [t for inst in range(N_INSTANCES) for t in results[(inst, k)]['times']]
    print(f'{k:5d}  {np.mean(all_vals):10.2f}  {np.std(all_vals):8.2f}  '
          f'{np.min(all_vals):8.2f}  {np.max(all_vals):8.2f}  '
          f'{np.mean(all_times)*1000:12.1f}')

## 6. Conclusion

### Interprétation des résultats

**Qualité :** On observe que pour un budget total fixé :
- k=1 (1 long run) explore davantage l'espace grâce à une température plus lente
  mais peut rester bloqué dans un bassin d'attraction local.
- k grand (beaucoup de runs courts) explore différentes régions de l'espace mais
  chaque run a peu de temps pour descendre vers un minimum local de qualité.

**Variabilité :** L'écart-type mesure la robustesse de la stratégie. Un écart-type
élevé indique que les résultats varient beaucoup d'une expérience à l'autre.
Un écart-type faible indique une convergence reproductible.

**Temps :** Le temps total reste comparable entre stratégies pour un budget fixé,
car le coût dominant est le nombre total d'évaluations de la fonction objectif.

### Peut-on conclure ?

Si l'écart entre les moyennes de différentes stratégies est inférieur à 1–2 fois
l'écart-type, la différence n'est **pas statistiquement significative** avec ce nombre
de répétitions. On pourrait appliquer un test de Mann-Whitney pour valider.

En pratique, la littérature (Hoos & Stützle, 2004) suggère que pour les problèmes
combinatoires, la stratégie **k modéré (5–10 runs)** offre souvent le meilleur
compromis : elle évite les mauvais optimaux locaux tout en bénéficiant d'un
refroidissement suffisamment lent dans chaque run.